# Director-AI — Streaming Halt Live Demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/colab_streaming_halt_demo.ipynb)

**Director-AI** halts LLM output mid-stream when coherence degrades. This notebook
demonstrates the three independent halt mechanisms with real-time matplotlib
visualisation and colour-coded token rendering.

No GPU, no API keys, no external services required. Runs entirely locally with
the heuristic scorer (word-overlap). For production accuracy, install `director-ai[nli]`
to enable the 0.4B DeBERTa NLI model.

**Three halt mechanisms:**
1. **Hard limit** — single token drops below absolute floor → immediate halt
2. **Sliding window** — rolling average of last N tokens falls below threshold
3. **Downward trend** — coherence drops by Δ over a window → gradient halt

[ANULUM Institute](https://www.anulum.li) · [Documentation](https://anulum.github.io/director-ai) · [GitHub](https://github.com/anulum/director-ai)

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-or-later
# Commercial license available
# © Concepts 1996–2026 Miroslav Šotek. All rights reserved.
# © Code 2020–2026 Miroslav Šotek. All rights reserved.
# ORCID: 0009-0009-3560-0851
# Contact: www.anulum.li | protoscience@anulum.li
# Director-Class AI — Streaming Halt Colab Demo

# Install Director-AI (heuristic-only for zero-dependency demo)
# For NLI-powered scoring: pip install "director-ai[nli]"
!pip install -q director-ai

In [ ]:
"""Imports and visualisation helpers."""

from __future__ import annotations

import logging
import textwrap

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from director_ai.core import StreamingKernel, StreamSession

# Suppress internal logs for clean demo output
logging.getLogger("DirectorAI").setLevel(logging.WARNING)


def plot_coherence_timeline(
    session: StreamSession,
    title: str,
    hard_limit: float,
    window_threshold: float,
    figsize: tuple[float, float] = (14, 4),
) -> None:
    """Matplotlib coherence timeline with halt zones and token labels."""
    fig, ax = plt.subplots(figsize=figsize)

    indices = list(range(len(session.events)))
    scores = [ev.coherence for ev in session.events]
    tokens = [ev.token.strip() or ev.token for ev in session.events]

    # Colour each bar by coherence level
    colours = []
    for i, ev in enumerate(session.events):
        if ev.halted:
            colours.append("#DC2626")  # red — halt token
        elif ev.coherence < hard_limit:
            colours.append("#EF4444")  # light red — below hard limit
        elif ev.coherence < window_threshold:
            colours.append("#F59E0B")  # amber — warning zone
        else:
            colours.append("#10B981")  # green — safe

    ax.bar(indices, scores, color=colours, width=0.7, edgecolor="white", linewidth=0.5)

    # Threshold lines
    ax.axhline(y=hard_limit, color="#DC2626", linestyle="--", linewidth=1.2,
               label=f"hard_limit = {hard_limit}")
    ax.axhline(y=window_threshold, color="#F59E0B", linestyle="--", linewidth=1.2,
               label=f"window_threshold = {window_threshold}")

    # Halt marker
    if session.halted:
        ax.axvline(x=session.halt_index, color="#DC2626", linewidth=2, alpha=0.5)
        ax.annotate(
            f"HALT\n{session.halt_reason}",
            xy=(session.halt_index, scores[session.halt_index]),
            xytext=(session.halt_index + 0.5, max(scores) * 0.85),
            fontsize=8,
            color="#DC2626",
            fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#DC2626", lw=1.5),
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#FEE2E2", edgecolor="#DC2626"),
        )

    # Token labels on x-axis
    ax.set_xticks(indices)
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Coherence Score")
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def render_token_stream(session: StreamSession, hard_limit: float) -> None:
    """HTML-rendered token stream with colour-coded coherence."""
    html_parts = ['<div style="font-family: monospace; font-size: 14px; '
                  'line-height: 2.0; padding: 12px; background: #1a1a2e; '
                  'border-radius: 8px; color: #eee;">']
    html_parts.append('<span style="color: #888; font-size: 11px;">'
                      'LLM output: </span>')

    for ev in session.events:
        if ev.halted:
            bg, fg = "#DC2626", "#fff"
        elif ev.coherence < hard_limit:
            bg, fg = "#7F1D1D", "#FCA5A5"
        elif ev.coherence < hard_limit + 0.15:
            bg, fg = "#78350F", "#FDE68A"
        else:
            bg, fg = "transparent", "#10B981"

        token_escaped = ev.token.replace("<", "&lt;").replace(">", "&gt;")
        tooltip = f"[{ev.index}] score={ev.coherence:.3f}"
        html_parts.append(
            f'<span title="{tooltip}" style="background:{bg}; color:{fg}; '
            f'padding: 1px 2px; border-radius: 3px;">{token_escaped}</span>'
        )

    if session.halted:
        html_parts.append(
            '<br><span style="color: #DC2626; font-weight: bold; '
            f'font-size: 12px;">⛔ HALTED — {session.halt_reason}</span>'
        )
    else:
        html_parts.append(
            '<br><span style="color: #10B981; font-weight: bold; '
            'font-size: 12px;">✓ APPROVED</span>'
        )

    stats = (f"tokens={session.token_count}  avg={session.avg_coherence:.3f}  "
             f"min={session.min_coherence:.3f}  "
             f"duration={session.duration_ms:.1f}ms")
    html_parts.append(f'<br><span style="color: #666; font-size: 10px;">{stats}</span>')
    html_parts.append("</div>")
    display(HTML("".join(html_parts)))

In [ ]:
def run_scenario(
    label: str,
    tokens: list[str],
    scores: list[float],
    hard_limit: float = 0.35,
    window_size: int = 5,
    window_threshold: float = 0.45,
    trend_window: int = 4,
    trend_threshold: float = 0.20,
) -> StreamSession:
    """Run a streaming scenario and render both visualisations."""
    kernel = StreamingKernel(
        hard_limit=hard_limit,
        window_size=window_size,
        window_threshold=window_threshold,
        trend_window=trend_window,
        trend_threshold=trend_threshold,
    )

    idx = 0
    def score_fn(_accumulated: str) -> float:
        nonlocal idx
        s = scores[min(idx, len(scores) - 1)]
        idx += 1
        return s

    session = kernel.stream_tokens(iter(tokens), score_fn)

    render_token_stream(session, hard_limit)
    plot_coherence_timeline(session, label, hard_limit, window_threshold)

    return session

## Scenario 1: Truthful Response → APPROVED

All tokens maintain high coherence. No halt mechanism triggers.
The response passes through unchanged.

In [ ]:
s1 = run_scenario(
    "Scenario 1: Truthful Response — All Checks Pass",
    tokens=[
        "Water", " boils", " at", " 100", " degrees", " Celsius",
        " (212", " °F)", " at", " standard", " atmospheric", " pressure",
        ".", " This", " is", " a", " well", "-established", " physical",
        " constant", ".",
    ],
    scores=[
        0.92, 0.90, 0.88, 0.91, 0.93, 0.90, 0.88, 0.87, 0.89, 0.91,
        0.90, 0.89, 0.90, 0.88, 0.87, 0.89, 0.91, 0.90, 0.88, 0.90,
        0.91,
    ],
)
print(f"Halted: {s1.halted}  |  Tokens: {s1.token_count}/21  |  Avg: {s1.avg_coherence:.3f}")

## Scenario 2: Blatant Hallucination → Hard Limit Halt

The LLM starts correctly but then claims a physically impossible temperature.
A single catastrophic token drops below `hard_limit=0.35` and the kernel
fires **immediately** — no averaging, no waiting.

In [ ]:
s2 = run_scenario(
    "Scenario 2: Blatant Hallucination — Hard Limit Fires",
    tokens=[
        "Water", " boils", " at", " 100", " degrees", " Celsius", ".",
        " But", " the", " real", " temperature", " is", " negative",
        " forty", " degrees", ".",
    ],
    scores=[
        0.92, 0.90, 0.91, 0.89, 0.88, 0.87, 0.86, 0.85, 0.84, 0.83,
        0.30,  # <-- hard_limit fires (0.30 < 0.35)
        0.15, 0.10, 0.08, 0.05, 0.03,
    ],
)
print(f"Halted: {s2.halted}  |  Token {s2.halt_index}/16  |  Reason: {s2.halt_reason}")

## Scenario 3: Gradual Drift → Downward Trend Halt

The response starts accurate, then slowly drifts into fabrication. No single
token is catastrophic, but the coherence slope over 4 tokens exceeds
`trend_threshold=0.20`. The kernel catches the drift before the output
becomes fully incoherent.

In [ ]:
s3 = run_scenario(
    "Scenario 3: Gradual Drift — Downward Trend Fires",
    tokens=[
        "Water", " boils", " at", " 100", " C.", " However", " at",
        " high", " altitude", " it", " actually", " boils", " at",
        " only", " 50", " C,", " which", " means", " climbers", " can",
        " boil", " water", " with", " body", " heat", " alone", ".",
    ],
    scores=[
        0.91, 0.89, 0.87, 0.90, 0.88, 0.78, 0.72, 0.65, 0.58, 0.52,
        0.46, 0.41, 0.38, 0.33, 0.28, 0.22, 0.18, 0.15, 0.12, 0.10,
        0.08, 0.05, 0.03, 0.02, 0.01, 0.01, 0.01,
    ],
)
print(f"Halted: {s3.halted}  |  Token {s3.halt_index}/27  |  Reason: {s3.halt_reason}")

## Scenario 4: Medical Hallucination — Dangerous Dosage

A medical LLM confidently outputs a drug dosage, then fabricates a dangerous
10× overdose. The hard limit fires instantly — in a clinical pipeline this
prevents a hallucinated dosage from reaching a patient.

In [ ]:
s4 = run_scenario(
    "Scenario 4: Medical — Dangerous Dosage Hallucination",
    tokens=[
        "The", " recommended", " dose", " of", " metformin", " is",
        " 500", "mg", " twice", " daily", ".", " However", ",",
        " some", " patients", " benefit", " from", " 5000", "mg",
        " as", " a", " single", " dose", ".",
    ],
    scores=[
        0.95, 0.93, 0.92, 0.94, 0.91, 0.90, 0.92, 0.91, 0.89, 0.88,
        0.87, 0.75, 0.70, 0.55, 0.48, 0.42, 0.38,
        0.12,  # <-- 5000mg: catastrophic hallucination
        0.08, 0.05, 0.03, 0.02, 0.01, 0.01,
    ],
    hard_limit=0.30,
    window_threshold=0.50,
)
print(f"Halted: {s4.halted}  |  Token {s4.halt_index}/24  |  Reason: {s4.halt_reason}")

## Scenario 5: Real Scorer — Heuristic Mode

Previous scenarios used synthetic scores to isolate halt mechanics. Now we use
the actual `CoherenceScorer` in heuristic mode (no NLI model required) to score
a prompt/response pair with ground truth facts. The scorer computes word-overlap
coherence in real time — this is what happens in production.

In [ ]:
from director_ai.core import CoherenceScorer, GroundTruthStore

# Set up ground truth facts
store = GroundTruthStore()
store.add("refund_policy", "Refunds are available within 30 days of purchase only.")
store.add("shipping", "Standard shipping takes 5-7 business days.")
store.add("returns", "Items must be unused and in original packaging.")

# Create scorer with heuristic backend (no NLI model needed)
scorer = CoherenceScorer(
    threshold=0.4,
    use_nli=False,
    ground_truth_store=store,
)

# Simulate a hallucinating customer support bot
prompt = "What is the refund policy?"
hallucinating_response = (
    "Our refund policy allows returns within 30 days. "
    "However we also offer a lifetime guarantee on all products "
    "and free return shipping worldwide with full refund "
    "plus a 20% bonus credit on your next purchase."
)

# Score the full response
_, cs = scorer.review(prompt, hallucinating_response)
print(f"Full response coherence: {cs.score:.3f}")
print(f"Approved: {cs.approved}")
print(f"h_logical: {cs.h_logical:.3f}  h_factual: {cs.h_factual:.3f}")

# Now stream it token by token through the kernel
kernel = StreamingKernel(
    hard_limit=0.25,
    window_size=5,
    window_threshold=0.40,
    trend_window=4,
    trend_threshold=0.20,
)

tokens = hallucinating_response.split(" ")
tokens = [(" " + t if i > 0 else t) for i, t in enumerate(tokens)]

def real_scorer_callback(accumulated_text: str) -> float:
    """Score accumulated text against ground truth."""
    _, score_obj = scorer.review(prompt, accumulated_text)
    return score_obj.score

session = kernel.stream_tokens(iter(tokens), real_scorer_callback)
render_token_stream(session, hard_limit=0.25)
plot_coherence_timeline(
    session,
    "Scenario 5: Real Heuristic Scorer — Customer Support Hallucination",
    hard_limit=0.25,
    window_threshold=0.40,
)

if session.halted:
    print(f"\nHalted at token {session.halt_index}/{len(tokens)}: {session.halt_reason}")
    print(f"Partial output preserved: {session.output!r}")
else:
    print(f"\nPassed — avg coherence: {session.avg_coherence:.3f}")

## Scenario 6: One-Shot Scoring with `score()`

The simplest API — score a prompt/response pair in one call. No streaming,
no kernel setup. Returns a `CoherenceScore` with composite score, approval
verdict, and diagnostic breakdowns.

In [ ]:
from director_ai import score

# Truthful response
cs_good = score(
    "What is the refund policy?",
    "Refunds are available within 30 days of purchase.",
    facts={"refund": "Refunds are available within 30 days of purchase only."},
    threshold=0.3,
)
print(f"Truthful:      score={cs_good.score:.3f}  approved={cs_good.approved}")

# Hallucinated response
cs_bad = score(
    "What is the refund policy?",
    "We offer lifetime unlimited refunds with free shipping and 50% bonus credit.",
    facts={"refund": "Refunds are available within 30 days of purchase only."},
    threshold=0.3,
)
print(f"Hallucinated:  score={cs_bad.score:.3f}  approved={cs_bad.approved}")

# Visualise comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for ax, cs_obj, label, colour in [
    (axes[0], cs_good, "Truthful", "#10B981"),
    (axes[1], cs_bad, "Hallucinated", "#DC2626"),
]:
    bars = ax.barh(
        ["Composite", "h_logical", "h_factual"],
        [cs_obj.score, cs_obj.h_logical, cs_obj.h_factual],
        color=colour,
        height=0.5,
    )
    ax.set_xlim(0, 1)
    ax.set_title(f"{label} (approved={cs_obj.approved})", fontweight="bold")
    ax.axvline(x=0.3, color="#888", linestyle="--", linewidth=1, label="threshold=0.3")
    ax.legend(fontsize=8)
    for bar in bars:
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
                f"{bar.get_width():.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Scenario 7: Side-by-Side — All Three Halt Mechanisms

Overlay all three scenarios on a single coherence plot to compare how different
degradation patterns trigger different halt mechanisms at different points.

In [ ]:
# Overlay comparison
fig, ax = plt.subplots(figsize=(14, 5))

scenarios = [
    (s1, "Truthful (approved)", "#10B981", "-"),
    (s2, "Hard limit halt", "#DC2626", "--"),
    (s3, "Trend halt", "#F59E0B", "-."),
]

for session, label, colour, ls in scenarios:
    scores_list = [ev.coherence for ev in session.events]
    ax.plot(range(len(scores_list)), scores_list, color=colour,
            linewidth=2, linestyle=ls, label=label, marker="o", markersize=4)
    if session.halted:
        ax.axvline(x=session.halt_index, color=colour, linewidth=1.5,
                   alpha=0.4, linestyle=":")
        ax.scatter([session.halt_index],
                   [scores_list[session.halt_index]],
                   color=colour, s=120, zorder=5, marker="X",
                   edgecolors="white", linewidth=1.5)

ax.axhline(y=0.35, color="#DC2626", linestyle="--", linewidth=1, alpha=0.5,
           label="hard_limit=0.35")
ax.axhline(y=0.45, color="#F59E0B", linestyle="--", linewidth=1, alpha=0.5,
           label="window_threshold=0.45")
ax.fill_between(range(30), 0, 0.35, alpha=0.05, color="#DC2626")
ax.fill_between(range(30), 0.35, 0.45, alpha=0.05, color="#F59E0B")

ax.set_xlabel("Token Index")
ax.set_ylabel("Coherence Score")
ax.set_title("Comparison: Three Halt Mechanisms", fontweight="bold", fontsize=13)
ax.set_ylim(0, 1.05)
ax.set_xlim(-0.5, max(len(s1.events), len(s2.events), len(s3.events)) + 0.5)
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Parameter Tuning Reference

| Parameter | Default | Effect |
|-----------|---------|--------|
| `hard_limit` | 0.5 | Absolute coherence floor — halt instantly if any token drops below |
| `window_size` | 10 | Number of tokens in sliding average window |
| `window_threshold` | 0.55 | Halt if sliding window average drops below this |
| `trend_window` | 5 | Tokens to measure coherence slope over |
| `trend_threshold` | 0.15 | Halt if slope drop exceeds this over trend_window |
| `halt_mode` | `"hard"` | `"hard"` = instant halt, `"soft"` = finish current sentence |
| `score_every_n` | 1 | Score every N-th token (reduce for speed, increase for accuracy) |

**Domain recommendations:**
- **Medical/Legal:** `hard_limit=0.30, window_threshold=0.50` — aggressive, minimise false negatives
- **Customer support:** `hard_limit=0.35, window_threshold=0.45` — balanced
- **Creative writing:** `hard_limit=0.15, window_threshold=0.30` — permissive, hallucination is a feature

---

**Next steps:**
- Install `director-ai[nli]` for 0.4B DeBERTa NLI scoring (75.8% balanced accuracy)
- Add your knowledge base via `GroundTruthStore` for RAG-grounded fact-checking
- Use `guard(client)` to wrap your OpenAI/Anthropic/Bedrock/Gemini/Cohere SDK

[Full documentation](https://anulum.github.io/director-ai) · [GitHub](https://github.com/anulum/director-ai) · [PyPI](https://pypi.org/project/director-ai/)

---
*Director-AI — [ANULUM Institute](https://www.anulum.li) / Fortis Studio, Marbach SG, Switzerland*